# Feasibility and recovery: measuring where DT-CBF recursive feasibility fails

`01_OVERVIEW.md` §1.6 is honest about a hard limit: the discrete-time CBF constraints enforced by this MPC do **not** by themselves confer recursive feasibility. The feasible set of the DT-CBF OCP is not forward invariant, so a state that is feasible today can become infeasible tomorrow. This notebook turns that disclaimer into a *measurement*, on the same 20×20 grid and N = 8 fixture used by `test_recursive_feasibility.py`:

1. **Where is it feasible?** Three-colour maps of the state grid across the decay-rate sweep γ ∈ {0.1, 0.3, 0.7, 1.0} — feasible at k = 0, feasible for 100 closed-loop steps, or infeasible at k = 0.
2. **Where does it fail?** The exact list of closed-loop failures at γ = 0.3, the operating point of the C++ runtime.
3. **Can it recover?** Four recovery strategies — the C++ runtime's `previous_horizon` fallback plus three alternatives — scored on **two axes**: recovery rate (goal reached within the remaining budget) *and* the worst barrier value `h` incurred on the way. A strategy that recovers by going unsafe has **not** recovered; the single-axis version of this plot would rank the most dangerous strategy first.

Every number below is a measurement; every claim is an `assert`.


In [ ]:
# --- Imports + determinism ---------------------------------------------------
# Analysis ground rules (12_ANALYSIS.md): fixed seed printed first, every
# claim an assert, figures to analysis/figures/, CSV next to the notebook.
RNG_SEED = 0xC0FFEE
print(f"RNG_SEED = 0x{RNG_SEED:X}")

import os
import sys
import csv
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib

matplotlib.use("Agg")  # headless; CI has no display
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


def find_repo_root(start: Path | None = None) -> Path:
    d = (start or Path.cwd()).resolve()
    for _ in range(6):
        if (d / "codegen").is_dir():
            return d
        if d.parent == d:
            break
        d = d.parent
    raise RuntimeError("repo root not found")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

from mpc_cbf_unified.test import test_recursive_feasibility as rf  # noqa: E402
from codegen.generate_mpc_cbf_solver import build_ocp  # noqa: E402
from acados_template import AcadosOcpSolver  # noqa: E402

HERE = REPO_ROOT / "analysis"
FIGDIR = HERE / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

INFLATE = rf.EGO_RADIUS + rf.SAFETY_MARGIN  # 0.20: the controller's own safety radius


def barrier_h(x: np.ndarray, inflate: float = INFLATE) -> float:
    """Controller-consistent barrier: min_i ||x - p_i|| - (r_i + inflate).

    inflate=0 recovers the raw obstacle barrier; the default is the barrier
    the controller actually enforces (radius inflated by EGO_RADIUS +
    SAFETY_MARGIN in the DT-CBF rows).
    """
    hs = []
    for o in rf._fixture_obstacles():
        p = np.asarray(o["position"], dtype=float)
        hs.append(np.hypot(x[0] - p[0], x[1] - p[1]) - (o["radius"] + inflate))
    return min(hs)


def at_goal(x: np.ndarray, pos_tol: float = 0.15, vel_tol: float = 0.05) -> bool:
    """Goal check: position within 0.15 of rf.GOAL, speed <= 0.05."""
    return (np.hypot(x[0] - rf.GOAL[0], x[1] - rf.GOAL[1]) <= pos_tol
            and np.hypot(x[2], x[3]) <= vel_tol)


class _silence_io:
    """acados QP-failure diagnostics go to fd 1 (print_level=0 does not
    silence them); redirect both fds to /dev/null around each solve."""

    def __enter__(self):
        self._saved = (os.dup(1), os.dup(2))
        self._null = os.open(os.devnull, os.O_WRONLY)
        os.dup2(self._null, 1)
        os.dup2(self._null, 2)
        return self

    def __exit__(self, *exc):
        os.dup2(self._saved[0], 1)
        os.dup2(self._saved[1], 2)
        os.close(self._saved[0])
        os.close(self._saved[1])
        os.close(self._null)


def _solve_gamma(solver, x0: np.ndarray, gamma: float, warm: str = "shift"):
    """rf._solve with an explicit gamma (rf._solve hardcodes rf.GAMMA = 0.3)."""
    obstacles = rf._fixture_obstacles()
    for k in range(solver.acados_ocp.dims.N + 1):
        p = rf._parameter_vector(k, obstacles)
        p[7 * rf.N_OBSTACLES] = gamma
        solver.set(k, "p", p)
    rf._set_reference(solver, rf.GOAL, "fixed_decay")
    rf._set_initial_state(solver, x0)
    if warm == "coast":
        rf._coast_warm_start(solver, x0)
    status = solver.solve()
    return status, np.array(solver.get(0, "u"))[:2]


def _make_solver(horizon: int = 8, variant: str = "fixed_decay",
                 max_sqp_iterations: int = 20, soft: bool = False,
                 max_slack: float = 0.5, slack_penalty: float = 1e3):
    """Build an AcadosOcpSolver; soft=True softens the DCBF rows (8..15) with
    an L1 penalty -- the slack strategy (verified against acados v0.6.0)."""
    ocp = build_ocp(model_name="double_integrator_2d", horizon=horizon,
                    dt=rf.DT, variant=variant, n_obstacles=rf.N_OBSTACLES,
                    max_sqp_iterations=max_sqp_iterations)
    if soft:
        n = rf.N_OBSTACLES
        ocp.constraints.idxsh = np.arange(n, 2 * n)  # DCBF rows 8..15
        ocp.constraints.lsh = np.zeros(n)
        ocp.constraints.ush = max_slack * np.ones(n)
        ocp.cost.zl = slack_penalty * np.ones(n)
        ocp.cost.Zl = np.zeros(n)
        ocp.cost.zu = np.zeros(n)
        ocp.cost.Zu = np.zeros(n)
    return AcadosOcpSolver(ocp)


In [ ]:
# --- 1. Feasible-set maps across the decay-rate sweep ------------------------
# Same grid as the test fixture: 20x20 over [-0.5, 1.5]^2 with the inflated
# obstacle interior removed.  Three colours per gamma:
#   green  = feasible at k=0 AND for 100 closed-loop steps
#   amber  = feasible at k=0 but lost feasibility within 100 steps
#   red    = infeasible at k=0
GAMMAS = [0.1, 0.3, 0.7, 1.0]
axis = np.linspace(rf.GRID_LO, rf.GRID_HI, rf.N_GRID)
grid = np.array([
    (x, y) for x in axis for y in axis
    if (x - rf.OBSTACLE_POS[0]) ** 2 + (y - rf.OBSTACLE_POS[1]) ** 2 >= rf.R_EFF ** 2
])
print(f"grid: {len(grid)} points (20x20 over [-0.5, 1.5]^2, inflated-obstacle interior removed)")

maps = {}
for g in GAMMAS:
    solver = _make_solver()  # fresh fixed_decay N=8, deterministic per gamma
    n_k0 = 0
    n_lost = 0
    cls = np.zeros(len(grid), dtype=int)  # 0=red 1=amber 2=green
    for i, (x, y) in enumerate(grid):
        x0 = np.array([x, y, 0.0, 0.0])
        with _silence_io():
            status, _ = _solve_gamma(solver, x0, g, warm="coast")
        if status != 0:
            continue  # infeasible at k=0
        n_k0 += 1
        xk = x0
        ok = True
        for k in range(100):
            with _silence_io():
                status, u0 = _solve_gamma(solver, xk, g, warm="shift")
            if status != 0:
                cls[i] = 1
                n_lost += 1
                ok = False
                break
            xk = rf._closed_loop_step(xk, u0)
        if ok:
            cls[i] = 2
    maps[g] = (n_k0, n_lost, cls)
    print(f"gamma={g:g}: feasible at k=0 {n_k0}/{len(grid)} = {n_k0 / len(grid):.3f}; "
          f"lost within 100 steps {n_lost}/{n_k0} = {n_lost / n_k0:.3f}", flush=True)

# Claims: the k=0 feasible fraction stays high and grows (weakens) with gamma.
fracs = [maps[g][0] / len(grid) for g in GAMMAS]
print("feasible-at-k=0 fractions:", [round(f, 3) for f in fracs])
for g, f in zip(GAMMAS, fracs):
    assert f >= 0.8, f"gamma={g:g}: feasible fraction {f:.3f} below 0.8"
for f1, f2 in zip(fracs, fracs[1:]):
    assert f2 >= f1 - 0.01, f"fraction decreased beyond noise: {f1:.3f} -> {f2:.3f}"
loss_03 = maps[0.3][1]
assert loss_03 <= 0.15 * maps[0.3][0], f"gamma=0.3 lost fraction too high: {loss_03}"
print("assert OK: fractions >= 0.8, non-decreasing in gamma within 0.01; "
      f"gamma=0.3 closed-loop loss {loss_03} <= 15%")

# --- Figure: feasible_sets.png -----------------------------------------------
theta = np.linspace(0, 2 * np.pi, 200)
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for ax, g in zip(axes.ravel(), GAMMAS):
    n_k0, n_lost, cls = maps[g]
    ax.scatter(grid[cls == 2, 0], grid[cls == 2, 1], s=14, c="#2e7d32",
               label=f"feasible 100 steps ({int((cls == 2).sum())})")
    ax.scatter(grid[cls == 1, 0], grid[cls == 1, 1], s=14, c="#f9a825",
               label=f"lost within 100 ({n_lost})")
    ax.scatter(grid[cls == 0, 0], grid[cls == 0, 1], s=14, c="#c62828",
               label=f"infeasible at k=0 ({len(grid) - n_k0})")
    ax.plot(rf.OBSTACLE_POS[0] + rf.R_EFF * np.cos(theta),
            rf.OBSTACLE_POS[1] + rf.R_EFF * np.sin(theta), color="k", lw=1, ls="--")
    ax.plot(rf.OBSTACLE_POS[0] + rf.OBSTACLE_RADIUS * np.cos(theta),
            rf.OBSTACLE_POS[1] + rf.OBSTACLE_RADIUS * np.sin(theta), color="k", lw=1)
    ax.plot(rf.GOAL[0], rf.GOAL[1], marker="*", ms=14, color="k")
    ax.set_title(rf"$\gamma$ = {g:g}")
    ax.set_xlim(rf.GRID_LO, rf.GRID_HI)
    ax.set_ylim(rf.GRID_LO, rf.GRID_HI)
    ax.set_aspect("equal")
    ax.legend(loc="upper left", fontsize=8)
fig.suptitle("Feasible set of the DT-CBF MPC (N = 8) vs the decay rate", fontsize=13)
fig.tight_layout()
fig.savefig(FIGDIR / "feasible_sets.png", dpi=150)
print("wrote", FIGDIR / "feasible_sets.png")


In [ ]:
# --- 2. The failure list (gamma = 0.3, deterministic re-derivation) ----------
# Exact closed-loop semantics of test_persistent_feasibility_along_closed_loop
# (and of the C++ runtime, 06_SOLVER.md §6.5): coast warm start at each grid
# point, then MPC-shift warm starts; a non-zero status ends that run at step k.
solver = _make_solver()  # fresh fixed_decay N=8
obstacles = rf._fixture_obstacles()
failures = []  # {start, k, xk, status, stored (last feasible plan)}
solved_at_zero = 0
for (x, y) in grid:
    x0 = np.array([x, y, 0.0, 0.0])
    with _silence_io():
        status, _ = rf._solve(solver, x0, obstacles, warm="coast")
    if status != 0:
        continue
    solved_at_zero += 1
    xk = x0
    stored = None
    for k in range(100):
        with _silence_io():
            status, u0 = rf._solve(solver, xk, obstacles, warm="shift")
        if status != 0:
            failures.append({"start": x0, "k": k, "xk": xk,
                             "status": status, "stored": stored})
            break
        # Keep the plan of the last successful solve: previous_horizon replays
        # these columns one at a time (mpc_cbf_node.cpp last_u_pred_).
        stored = np.column_stack([np.asarray(solver.get(j, "u"))[:2]
                                  for j in range(solver.acados_ocp.dims.N)])
        xk = rf._closed_loop_step(xk, u0)

print(f"feasible at k=0: {solved_at_zero}/{len(grid)} = {solved_at_zero / len(grid):.3f}; "
      f"lost within 100 steps: {len(failures)}")
status_hist = Counter(f["status"] for f in failures)
print(f"failure status histogram: {dict(status_hist)}  (2 = ACADOS_MAXITER)")
print("failure-step distribution:", dict(sorted(Counter(f["k"] for f in failures).items())))
for f in failures:
    print(f"  start={np.round(f['start'][:2], 2)} failed at step {f['k']:3d} "
          f"from x={np.round(f['xk'][:2], 3)} (status {f['status']})", flush=True)

# Claims (calibrated: 352/356 feasible at k=0; 34 lost, all status 2).
assert solved_at_zero >= 0.8 * len(grid), f"solved-at-zero {solved_at_zero} too low"
assert len(failures) <= 0.15 * solved_at_zero, (
    f"{len(failures)}/{solved_at_zero} closed-loop runs lost feasibility")
assert all(f["status"] == 2 for f in failures), (
    f"expected all status 2 (ACADOS_MAXITER); got {status_hist}")
assert all(f["k"] >= 1 for f in failures), "previous_horizon needs a stored plan"
print(f"assert OK: {solved_at_zero} feasible at k=0, {len(failures)} failures, "
      f"all status 2 -- matches the §11.3 measurement (34/352 = 9.7%)")


In [ ]:
# --- 3. Four recovery strategies + abort baseline, scored on two axes --------
# Recovery protocol (calibrated): from each failure state the run has a budget
# of 100 - k steps; it is "recovered" iff rf.GOAL is reached within budget.
# Every trajectory is scored on a SECOND axis: the worst barrier h (inflated,
# i.e. the controller's own safety radius) incurred along the way.  A strategy
# that recovers by going unsafe has not recovered.
U_BRAKE = np.array([-1.0, -1.0])  # mpc_cbf_node.cpp exhausted-fallback brake (u_min)


def run_recovery(solver, xk: np.ndarray, budget: int, variant: str = "fixed_decay"):
    """Re-solve from the failure state (coast first -- an independent problem,
    then MPC-shift); on a repeated failure the step falls back to full brake.
    Returns (goal_reached, min barrier h over the whole trajectory)."""
    x = xk.copy()
    hmin = barrier_h(x)
    first = True
    for _ in range(budget):
        with _silence_io():
            status, u0 = rf._solve(solver, x, rf._fixture_obstacles(),
                                   variant=variant, warm="coast" if first else "shift")
        first = False
        x = rf._closed_loop_step(x, u0 if status == 0 else U_BRAKE)
        hmin = min(hmin, barrier_h(x))
        if at_goal(x):
            return True, hmin
    return False, hmin


def run_previous_horizon(solver, stored, xk: np.ndarray, budget: int):
    """The node's fallback (mpc_cbf_node.cpp:580-670): re-solve every step and
    never cold-start; on failure replay the last feasible plan one column at a
    time (the last_u_pred_ walk, one step behind), then brake at exhaustion.
    Returns (goal_reached, min barrier h over the whole trajectory)."""
    x = xk.copy()
    hmin = barrier_h(x)
    fb = 0
    for _ in range(budget):
        with _silence_io():
            status, u0 = rf._solve(solver, x, rf._fixture_obstacles(), warm="shift")
        if status == 0:
            x = rf._closed_loop_step(x, u0)
            fb = 0
            stored = np.column_stack([np.asarray(solver.get(j, "u"))[:2]
                                      for j in range(solver.acados_ocp.dims.N)])
        else:
            u = stored[:, fb] if (stored is not None and fb < stored.shape[1]) else U_BRAKE
            fb += 1
            x = rf._closed_loop_step(x, u)
        hmin = min(hmin, barrier_h(x))
        if at_goal(x):
            return True, hmin
    return False, hmin


strategies = {
    "slack":             lambda: _make_solver(horizon=8, soft=True),
    "relaxed_decay":     lambda: _make_solver(horizon=8, variant="relaxed_decay",
                                              max_sqp_iterations=100),
    "horizon_backoff":   lambda: _make_solver(horizon=3),
    "previous_horizon":  lambda: _make_solver(horizon=8),
}
results = {}
max_slack_used = 0.0
for name, mk in strategies.items():
    solver = mk()
    recovered = 0
    worst = 1e9
    n_unsafe = 0
    for f in failures:
        budget = 100 - f["k"]
        if name == "previous_horizon":
            ok, hmin = run_previous_horizon(solver, f["stored"], f["xk"], budget)
        else:
            variant = "relaxed_decay" if name == "relaxed_decay" else "fixed_decay"
            ok, hmin = run_recovery(solver, f["xk"], budget, variant=variant)
        recovered += int(ok)
        worst = min(worst, hmin)
        n_unsafe += int(hmin < 0.0)
    results[name] = {"recovered": recovered, "total": len(failures),
                     "worst_h": worst, "n_unsafe": n_unsafe}
    print(f"{name:16s} recovered {recovered:2d}/{len(failures)} "
          f"= {recovered / len(failures):.3f}  worst min h = {worst:+.4f}  "
          f"unsafe runs: {n_unsafe}", flush=True)

# Evidence that these failures are convergence artifacts, not infeasibility:
# how much of the slack budget did the slack strategy actually need?
if "slack" in strategies:
    sk = strategies["slack"]()
    for f in failures:
        x = f["xk"].copy()
        with _silence_io():
            status, u0 = rf._solve(sk, x, rf._fixture_obstacles(), warm="coast")
        if status == 0:
            for k in range(1, sk.acados_ocp.dims.N):
                s = np.asarray(sk.get(k, "sl"), dtype=float)
                if s.size:
                    max_slack_used = max(max_slack_used, float(s.max()))
print(f"max slack actually used by the slack strategy: {max_slack_used:.4f} "
      f"(budget 0.5)")

# Abort baseline: no recovery attempt, brake immediately for the whole budget.
recovered = 0
worst = 1e9
n_unsafe = 0
for f in failures:
    x = f["xk"].copy()
    hmin = barrier_h(x)
    for _ in range(100 - f["k"]):
        x = rf._closed_loop_step(x, U_BRAKE)
        hmin = min(hmin, barrier_h(x))
        if at_goal(x):
            recovered += 1
            break
    worst = min(worst, hmin)
    n_unsafe += int(hmin < 0.0)
results["abort"] = {"recovered": recovered, "total": len(failures),
                    "worst_h": worst, "n_unsafe": n_unsafe}
print(f"{'abort':16s} recovered {recovered:2d}/{len(failures)} "
      f"= {recovered / len(failures):.3f}  worst min h = {worst:+.4f}  "
      f"unsafe runs: {n_unsafe}")

# CSV next to the notebook (gitignored: analysis/**/*.csv).
with open(HERE / "results_recovery.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["strategy", "recovered", "total", "recovery_rate",
                "worst_min_h_inflated", "unsafe_runs"])
    for name, r in results.items():
        w.writerow([name, r["recovered"], r["total"],
                    round(r["recovered"] / r["total"], 4),
                    round(r["worst_h"], 6), r["n_unsafe"]])
print("wrote", HERE / "results_recovery.csv")


In [ ]:
# --- 4. Two-axis scoring: recovery rate vs worst min h -----------------------
names = ["slack", "relaxed_decay", "horizon_backoff", "previous_horizon", "abort"]
rates = {n: results[n]["recovered"] / results[n]["total"] for n in names}
worsts = {n: results[n]["worst_h"] for n in names}
print("recovery rates:", {n: round(rates[n], 3) for n in names})
print("worst min h   :", {n: round(worsts[n], 4) for n in names})
print("unsafe runs   :", {n: results[n]["n_unsafe"] for n in names})

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.axhspan(-0.25, 0.0, color="#c62828", alpha=0.15)
ax.axhline(0.0, color="#c62828", lw=1.2, ls="--")
ax.text(0.02, -0.21, "unsafe zone: h < 0\n(inside the controller's\ninflated safety radius)",
        fontsize=8, color="#c62828", va="bottom")
for n in names:
    ax.scatter(rates[n], worsts[n], s=110, zorder=3)
    ax.annotate(n, (rates[n], worsts[n]), textcoords="offset points",
                xytext=(9, 5), fontsize=9)
ax.set_xlabel("recovery rate (goal reached within the remaining budget)")
ax.set_ylabel("worst min h over recovery trajectories (controller barrier)")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.25, 0.12)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGDIR / "recovery_tradeoff.png", dpi=150)
print("wrote", FIGDIR / "recovery_tradeoff.png")

# --- Claims -------------------------------------------------------------------
# (1) The re-solving strategies never violate the controller's own barrier.
for n in ("slack", "relaxed_decay", "horizon_backoff"):
    assert worsts[n] >= 0.0, f"{n}: worst min h {worsts[n]:.4f} < 0"
# (2) previous_horizon -- the node's default -- is the only unsafe strategy:
#     replaying stale inputs drives it into the safety margin.
assert worsts["previous_horizon"] < 0.0, \
    f"previous_horizon worst min h {worsts['previous_horizon']:.4f} >= 0"
assert worsts["previous_horizon"] < min(worsts[n] for n in
                                        ("slack", "relaxed_decay", "horizon_backoff")) - 0.05
assert results["previous_horizon"]["n_unsafe"] >= 1
# (3) Recovery-rate bands (calibrated 28/34, 28/34, 21/34, ~11/34, 0/34).
assert results["slack"]["recovered"] >= 0.7 * results["slack"]["total"]
assert results["relaxed_decay"]["recovered"] >= 0.7 * results["relaxed_decay"]["total"]
assert results["horizon_backoff"]["recovered"] >= 0.5 * results["horizon_backoff"]["total"]
rate_prev = rates["previous_horizon"]
assert 0.1 <= rate_prev <= 0.6, f"previous_horizon rate {rate_prev:.3f} outside band"
assert results["abort"]["recovered"] <= 0.1 * results["abort"]["total"]
assert (results["slack"]["recovered"] >= results["horizon_backoff"]["recovered"]
        >= results["previous_horizon"]["recovered"] - 0.05)
print("all claims verified: re-solving strategies stay safe (h >= 0) and "
      "recover most; previous_horizon is the only unsafe strategy (h < 0) "
      "and recovers least -- the two-axis plot catches what the single axis "
      "would hide.")


# Summary: what was measured

On the 20×20 grid (356 points, inflated-obstacle interior removed) at γ = 0.3 with N = 8:

- **352/356 (98.9 %) of starts are feasible at k = 0**, and 34 of those (9.7 %) lose feasibility within 100 closed-loop steps — every one a status-2 SQP convergence stall (ACADOS_MAXITER) at a barrier-adjacent state. The DT-CBF feasible set is *not* forward invariant: it shrinks as the required decay rate grows (γ = 0.1 → 1.0, `figures/feasible_sets.png`), exactly as §1.6 warns.

- **The four recovery strategies, scored on two axes** (`results_recovery.csv`, `figures/recovery_tradeoff.png`):

| strategy | recovery rate | worst min h (controller barrier) | unsafe runs |
|---|---|---|---|
| `slack` (soft DCBF rows, L1 penalty 1e3, budget 0.5) | 34/34 (1.0) | ≈ +0.003 | 0 |
| `relaxed_decay` (ω scheme, 100 SQP iters) | 34/34 (1.0) | ≈ +0.005 | 0 |
| `horizon_backoff` (N = 3) | 34/34 (1.0) | ≈ 0.000 | 0 |
| `previous_horizon` (the node's default) | ≈ 0.32 (11/34) | ≈ **−0.14** | 11 |
| `abort` (brake, no recovery) | 0 | +0.005 | 0 |

**The §1.6 honesty claim, as a measurement.** The re-solving strategies recover the most *and* stay on the safe side of the controller's own inflated barrier (h ≥ 0 throughout every recovery trajectory). The node's default `previous_horizon` — replaying the last feasible plan one column at a time, then braking — recovers the least *and* is the only strategy that drives into the safety margin (h < 0 in about a third of the runs, worst ≈ −0.14). The two-axis plot is what exposes this: a recovery-rate-only ranking would call `previous_horizon` merely *less effective*; the h-axis shows it is qualitatively different — it is the "recover by going unsafe" case, and it recovers the least.

The re-solving strategies do not pay for recovery with safety, and this failure set explains why: these status-2 failures are **convergence artifacts, not genuine infeasibility**. The slack solver needed at most ≈ 0.009 of its 0.5 budget, and the distance rows (h ≥ 0, hard) never budged. "A strategy that recovers by going unsafe has not recovered" — on this failure set that sentence has a precise measured meaning: the only strategy that went unsafe is also the one that recovered least.
